# 实践 16.2  同一育种群比较芯片、GBS、靶向面板和低深度全基因组测序

《统计遗传与智能育种学》第十六章 · 电子资源 `16b_genotyping_strategy.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：16b_genotyping_strategy.ipynb。实践用Markov单倍型生成沿染色体衰减的LD结构，并在同一真实基因型、30个QTL、性状和训练/测试划分上构造四种测量机制：SNP芯片偏好训练发现群中的常见变异；GBS具有固定可测位点集合和较高个体缺失；靶向面板含10个预先设定为已验证的功能QTL与170个骨架标记；1× lpWGS直接根据Poisson读段深度和测序错误计算单个位点后验剂量，先验频率由训练群真实基因型估计。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果见下方代码输出。QTL标记能力以训练集观测矩阵与QTL真值的最大相关平方评价，包含测量和填补误差。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1602；500个个体、1200个模拟SNP、30个真实QTL；训练350、测试150；同一遗传真值用于四种测量机制；无外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
from scipy.special import logsumexp
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 1602
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

# 用Markov单倍型形成沿染色体衰减的LD，再合并两条单倍型得到0/1/2基因型。
n, m = 500, 1200
def make_haplotype(n_hap):
    H = np.empty((n_hap, m), dtype=np.int8)
    p0 = 0.30
    H[:, 0] = rng.binomial(1, p0, n_hap)
    for j in range(1, m):
        # 约7%的位置发生状态更新；更新频率随位置缓慢变化。
        p_j = 0.18 + 0.22*(0.5 + 0.5*np.sin(j/85))
        switch = rng.random(n_hap) < 0.07
        H[:, j] = H[:, j-1]
        H[switch, j] = rng.binomial(1, p_j, switch.sum())
    return H
H = make_haplotype(2*n)
G = H[0::2].astype(float) + H[1::2].astype(float)
maf = np.minimum(G.mean(axis=0)/2, 1-G.mean(axis=0)/2)
usable = np.flatnonzero(maf >= 0.05)
qtl = rng.choice(usable, 30, replace=False)
eff = rng.normal(0, 0.35, len(qtl))
y = G[:, qtl] @ eff + rng.normal(0, 1.5, n)
tr = np.arange(350)
te = np.arange(350, n)
print('population/markers/QTL', n, m, len(qtl))
print('train/test', len(tr), len(te), 'median_MAF', round(float(np.median(maf)), 3))

# 四种测量机制。芯片位点按训练群MAF偏好选择，模拟常见的发现/设计偏倚。
discovery = tr[:120]
p_disc = G[discovery].mean(axis=0)/2
maf_disc = np.minimum(p_disc, 1-p_disc)
chip_pool = np.flatnonzero(maf_disc >= 0.20)
chip = np.sort(rng.choice(chip_pool, 300, replace=False))

# GBS：固定可切/可测位点集合 + 个体层面较高缺失。
gbs = np.sort(rng.choice(usable, 420, replace=False))

# 靶向panel：10个预设已验证功能位点 + 170个全基因组骨架。这里只给10/30个QTL先验知识。
validated = rng.choice(qtl, 10, replace=False)
backbone_pool = np.setdiff1d(usable, validated)
backbone = np.sort(rng.choice(backbone_pool, 170, replace=False))
panel = np.unique(np.r_[validated, backbone])

def measured_matrix(idx, miss_rate, stream_seed):
    X = G[:, idx].copy()
    rg = np.random.default_rng(stream_seed)
    X[rg.random(X.shape) < miss_rate] = np.nan
    return X
X_chip = measured_matrix(chip, 0.01, 16101)
X_gbs = measured_matrix(gbs, 0.22, 16102)
X_panel = measured_matrix(panel, 0.02, 16103)

# lpWGS：Poisson读深 + 1%测序错误 + 训练群频率HWE先验，计算sitewise posterior dosage。
SEQ_ERROR, LOWCOV_DEPTH = 0.01, 1.0
p_train = np.clip(G[tr].mean(axis=0)/2, 0.01, 0.99)
rg = np.random.default_rng(16104)
d = rg.poisson(LOWCOV_DEPTH, size=G.shape)
q_true = SEQ_ERROR + (1-2*SEQ_ERROR)*G/2
alt = rg.binomial(d, q_true)
prior = np.stack([(1-p_train)**2, 2*p_train*(1-p_train), p_train**2], axis=1)
q = np.array([SEQ_ERROR, 0.5, 1-SEQ_ERROR])
logpost = np.empty((n, m, 3))
for g in range(3):
    logpost[:, :, g] = alt*np.log(q[g]) + (d-alt)*np.log(1-q[g]) + np.log(prior[:, g])[None, :]
logpost -= logsumexp(logpost, axis=2, keepdims=True)
post = np.exp(logpost)
X_lowcov = post[:, :, 1] + 2*post[:, :, 2]
print('1x_lowcov_covered_ge1', round(float((d >= 1).mean()), 3))

# 在训练集上评价观测标记矩阵与QTL真值的最大相关平方，包含测量和填补误差。
def impute_train_mean(X):
    imp = SimpleImputer(strategy='mean')
    return imp.fit_transform(X[tr])

def qtl_tagging(X):
    Xt = impute_train_mean(X)
    Gq = G[tr][:, qtl]
    # 标准化后相关系数矩阵；常数列做安全处理。
    Xz = (Xt - Xt.mean(0)) / np.where(Xt.std(0) > 0, Xt.std(0), 1)
    Qz = (Gq - Gq.mean(0)) / np.where(Gq.std(0) > 0, Gq.std(0), 1)
    r2 = (Xz.T @ Qz / len(tr))**2
    best = r2.max(axis=0)
    return float(best.mean()), float((best >= 0.80).mean())

def prediction_r(X):
    model = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), Ridge(alpha=80.0))
    model.fit(X[tr], y[tr])
    return float(np.corrcoef(model.predict(X[te]), y[te])[0, 1])

strategies = [
    ('SNP_chip', X_chip),
    ('GBS', X_gbs),
    ('target_panel', X_panel),
    ('lpWGS_1x_sitewise_dosage', X_lowcov),
]
rows = []
for name, X in strategies:
    mean_r2, frac08 = qtl_tagging(X)
    rows.append({
        'strategy': name, 'markers': X.shape[1],
        'missing': np.isnan(X).mean(),
        'mean_best_QTL_r2': mean_r2,
        'QTL_frac_r2_ge_0.8': frac08,
        'test_r': prediction_r(X),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1602
population/markers/QTL 500 1200 30
train/test 350 150 median_MAF 0.306
1x_lowcov_covered_ge1 0.632
                strategy  markers  missing  mean_best_QTL_r2  QTL_frac_r2_ge_0.8  test_r
                SNP_chip      300     0.01             0.824               0.600   0.329
                     GBS      420     0.22             0.673               0.033   0.185
            target_panel      180     0.02             0.792               0.633   0.404
lpWGS_1x_sitewise_dosage     1200     0.00             0.384               0.000   0.325


结果解释：test r比较本例四种测量情景的预测表现，实际选型还需评价成本与稳定性。靶向面板预先知道10/30个已验证功能QTL，代表具有明确先验功能知识的情景，并未使用全部因果位点真值；芯片的常见变异偏好用于模拟发现偏倚（ascertainment bias）；GBS的缺失机制被显式建模。lpWGS虽然已经包含读段水平测量与后验剂量，但仍未利用参考单倍型LD，因此与正式基因型填充仍有明确差别。

可修改练习：把GBS缺失率设为0.10和0.35；把validated功能位点数从10改成0；把LOWCOV_DEPTH改成0.5和2.0。每次只改一个因素，重新运行并比较mean_best_QTL_r2、QTL_frac_r2_ge_0.8和test_r。